# Part 1 — Write functions for clarity

Before you write code, sketch what your program will do. Your answers to Project 01 and 02 may help.

You can decide how to break the program up.

### Goal: 
Learn “current note to likely next notes (with counts)” from example melodies (a bigram model), then generate new melodies by weighted random sampling.

### Design:

Parsing & preprocessing: parse_melody, add_start_end_tokens

Modeling: build_bigrams

Sampling & generation: weighted_choice, generate_melody

Display & analysis: print_bigram, most_common_transition

(Optional) Constraint to avoid 3 identical notes in a row

# Part 2 — Build the bigram model

Your program should be able to:
- Read a melody (lists of notes, ABC, MIDI, etc.). For reference, we will provide you a dataset of melodies represented by music notes. If you are interested in other representations, feel free to explore them.
- Build a bigram table showing “current note → next-note counts.”
- Generate new melodies using that model.
Your code must:
   - avoid long monolithic scripts,
   - be modularized and use functions meaningfully,
   - be easy for someone else to follow.
 
Below you will see two examples of melodies; both will be provided to you later as a dataset so that you can start generating melodies.

- **Example 1**: A sequence of notes without duration information: `E4 F#4 G4 A4 B4 A4 G4 F#4 E4`, where `#` represents one semitone up, `C4` denotes C in the 4th octave, `C5` denotes the C one octave higher than C4. Similar logic applies to other pitches.
- **Example 2**: A sequence of notes with duration information: `F4_1.3 A4_0.45 F4_0.45 A4_0.45 A#4_1.34 D5_0.45 A#4_0.45 D5_0.45 F5_2.23`, where `_x` represents `x` number of beats. You can convert beats to time by $time = 60*beats/tempo$, and feel free to define your own tempo.

In [1]:
from collections import Counter, defaultdict
from typing import List, Dict, Tuple
import random

Melody = List[str]
Bigram = Dict[str, Counter]

def parse_melody(line: str) -> Melody:
    """Split a space-separated melody line into tokens (supports duration tokens like F4_1.3)."""
    return [tok for tok in line.strip().split() if tok]

def add_start_end_tokens(melody: Melody) -> Melody:
    """Add start (^) and end ($) tokens so melodies start/stop naturally."""
    return ["^"] + melody + ["$"]

def build_bigrams(melodies: List[Melody]) -> Bigram:
    """Count all adjacent pairs (curr, next) across melodies to form a bigram frequency table."""
    model: Bigram = defaultdict(Counter)
    for m in melodies:
        for i in range(len(m) - 1):
            curr, nxt = m[i], m[i+1]
            model[curr][nxt] += 1
    return model


# Part 3 — Generate new melodies

* Start with a random note. While your melody isn’t long enough:
  * Look up for the next notes based on the current note.
  * Choose one at random, weighted by counts.
  * Append it to your melody.
* Print the melody as output.

In [2]:
def weighted_choice(counter: Counter) -> str:
    """Choose a successor weighted by counts."""
    keys = list(counter.keys())
    weights = [counter[k] for k in keys]
    return random.choices(keys, weights=weights, k=1)[0]

def violates_three_repeat(seq: Melody) -> bool:
    """Return True if the last 3 notes are identical."""
    return len(seq) >= 3 and seq[-1] == seq[-2] == seq[-3]

def generate_melody(model: Bigram, max_len: int = 32, forbid_three_repeats: bool = False) -> Melody:
    """Generate a melody from the bigram model; stop at '$' or length cap."""
    curr = "^"
    out: Melody = []
    for _ in range(max_len):
        if curr not in model or not model[curr]:
            break  # dead end
        nxt = weighted_choice(model[curr])
        if nxt == "$":
            break

        out.append(nxt)

        if forbid_three_repeats and violates_three_repeat(out):
            # Simple re-draw once to avoid a triple repeat; you could loop for robustness.
            alt = weighted_choice(model[curr])
            if alt != nxt:
                out[-1] = alt

        curr = out[-1]
    return out


# Part 4 — Show your results

* Print your bigram model in a readable way.
* Generate and print at least **3 sample melodies**.

Example output:

```
Bigram model:
C → {'D': 3, 'E': 1}
D → {'E': 2, 'G': 1}
...

Generated melodies:
1. C D E G C D
2. G F E D C E
3. E F G A C D
```
# Part 5 — Clean Up

* If we follow the bigram model, melody generation never stops. To prevent this situation,  add **start (`^`)** and **end (`$`)** tokens so melodies can stop naturally.
* Write a function to find the **most common transition** in the dataset.
* **Optional**: Add constraints: e.g., forbid repeating the same note 3+ times in a row. Feel free to define your own set of constraints.
* Test your new code and generate longer melodies and see if they sound “more musical” than unigram ones.

In [3]:
def print_bigram(model: Bigram) -> None:
    """Print successors sorted by count, for readability."""
    for curr, counter in model.items():
        items = dict(sorted(counter.items(), key=lambda kv: kv[1], reverse=True))
        print(f"{curr} \u2192 {items}")  # → arrow

def most_common_transition(model: Bigram) -> Tuple[str, str, int]:
    """Return (curr, next, count) for the single most frequent transition overall."""
    best = ("", "", -1)
    for curr, counter in model.items():
        if not counter:
            continue
        nxt, cnt = counter.most_common(1)[0]
        if cnt > best[2]:
            best = (curr, nxt, cnt)
    return best


In [4]:
if __name__ == "__main__":
    random.seed(42)  # reproducible demo, optional

    # Example 1: notes without duration
    line1 = "E4 F#4 G4 A4 B4 A4 G4 F#4 E4"
    # Example 2: notes with duration (treat whole token as a note for the first version)
    line2 = "F4_1.3 A4_0.45 F4_0.45 A4_0.45 A#4_1.34 D5_0.45 A#4_0.45 D5_0.45 F5_2.23"

    melodies_raw = [parse_melody(line1), parse_melody(line2)]
    melodies = [add_start_end_tokens(m) for m in melodies_raw]

    model = build_bigrams(melodies)

    print("Bigram model:")
    print_bigram(model)
    print()

    print("Generated melodies:")
    for i in range(1, 4):
        m = generate_melody(model, max_len=32, forbid_three_repeats=True)
        clean = [x for x in m if x not in {"^", "$"}]
        print(f"{i}.", " ".join(clean))
    print()

    c, n, k = most_common_transition(model)
    print(f"Most common transition: {c} -> {n} ({k})")


Bigram model:
^ → {'E4': 1, 'F4_1.3': 1}
E4 → {'F#4': 1, '$': 1}
F#4 → {'G4': 1, 'E4': 1}
G4 → {'A4': 1, 'F#4': 1}
A4 → {'B4': 1, 'G4': 1}
B4 → {'A4': 1}
F4_1.3 → {'A4_0.45': 1}
A4_0.45 → {'F4_0.45': 1, 'A#4_1.34': 1}
F4_0.45 → {'A4_0.45': 1}
A#4_1.34 → {'D5_0.45': 1}
D5_0.45 → {'A#4_0.45': 1, 'F5_2.23': 1}
A#4_0.45 → {'D5_0.45': 1}
F5_2.23 → {'$': 1}

Generated melodies:
1. F4_1.3 A4_0.45 F4_0.45 A4_0.45 A#4_1.34 D5_0.45 F5_2.23
2. E4 F#4 G4 F#4 G4 A4 G4 F#4 G4 F#4 E4 F#4 E4
3. E4 F#4 E4 F#4 G4 A4 G4 F#4 E4

Most common transition: ^ -> E4 (1)
